# STEP 2.0 GEE 外部数据增强
本模块先继承旧表中可严格匹配的 GEE 指标，再对缺失事件补充 ERA5-Land、GPW、GHSL 和世界银行国家背景指标；NASA POWER/MERRA-2 提供不依赖密钥的独立气象产品敏感性分析。JSON 密钥只通过环境变量读取。

In [1]:
from pathlib import Path # 导入跨平台路径工具
import os,sys # 导入环境变量与解释器路径模块
import pandas as pd # 导入表格处理模块
ROOT=Path.cwd().resolve() # 读取当前工作目录
ROOT=ROOT.parent if ROOT.name=='notebooks' else ROOT # 从notebooks目录回到项目根目录
sys.path.insert(0,str(ROOT)) # 将项目根目录加入模块搜索路径
from src.pipeline import merge_legacy_metrics,run_gee_enrichment,run_case_control_weather # 导入增量匹配、事件增强与病例交叉函数
from src.external import attach_country_context,attach_fire_service_capacity,download_world_bank_context,download_nasa_power_case_control,extract_ctif_fire_service_capacity,write_data_registry # 导入国家背景、消防能力、POWER敏感性与数据源登记函数
EVENTS=ROOT/'data/processed/events_standardised.csv' # 指定标准化事件表
LEGACY=ROOT/'data/interim/legacy_events_gee_enriched_20260826.csv' # 指定旧GEE结果快照
INCREMENTAL=ROOT/'data/interim/events_incremental_metrics.csv' # 指定增量合并输出
CHECKPOINT=ROOT/'data/interim/gee_checkpoint_current.csv' # 指定当前GEE检查点
FINAL=ROOT/'data/processed/events_enriched.csv' # 指定最终增强事件表
CALENDAR=ROOT/'data/interim/case_control_calendar.csv' # 指定病例交叉日期表
CASE_CONTROL=ROOT/'data/processed/case_control_weather.csv' # 指定GEE病例交叉气象暴露输出
POWER_CASE_CONTROL=ROOT/'data/processed/case_control_weather_nasa_power_v10.csv' # 指定NASA POWER v10病例交叉敏感性表
POWER_CACHE=ROOT/'data/external/nasa_power_daily' # 指定NASA POWER逐地点原始缓存目录
WORLD_BANK=ROOT/'data/interim/world_bank_country_year.csv' # 指定世界银行国家年份背景表
CTIF_SOURCE=Path(os.getenv('CTIF_REPORT_PATH',ROOT.parents[1]/'vibe_projects/全球建筑失火项目/CTIF_Report30.pdf')) # 指定本地CTIF第30号报告或读取环境变量覆盖路径
CTIF_CAPACITY=ROOT/'data/interim/ctif_fire_service_capacity_2010_2023.csv' # 指定CTIF国家消防服务能力派生表
DATA_REGISTRY=ROOT/'outputs/tables/data_source_registry.csv' # 指定外部数据源登记表

In [2]:
events=pd.read_csv(EVENTS,encoding='utf-8-sig',parse_dates=['event_date']) # 读取标准化事件表
events=merge_legacy_metrics(events,LEGACY) # 继承可严格匹配的旧GEE指标
events.to_csv(INCREMENTAL,index=False,encoding='utf-8-sig') # 保存增量匹配结果
print(events['metric_provenance'].value_counts(dropna=False)) # 显示旧指标继承与待处理数量

metric_provenance
legacy_exact_date_geo_match    148
requires_current_gee            91
Name: count, dtype: int64


In [3]:
RUN_GEE=os.getenv('RUN_GEE','0')=='1' # 仅在显式设置环境变量后调用GEE
current=run_gee_enrichment(events,CHECKPOINT,only_missing=True) if RUN_GEE else pd.DataFrame() # 批量补充当前缺失事件指标
if not current.empty: # 仅在存在当前GEE结果时进入增量更新
    current_index=current.set_index('event_id') # 将当前GEE结果按事件编号索引
    events_index=events.set_index('event_id') # 将事件主表按事件编号索引
    for column in current_index.columns: events_index.loc[current_index.index,column]=current_index[column] # 仅更新当前完成事件并保留旧严格匹配值
    events=events_index.reset_index() # 恢复事件编号普通字段
refresh_context=os.getenv('REFRESH_WORLD_BANK','0')=='1' # 读取是否刷新世界银行官方接口的开关
context=download_world_bank_context(WORLD_BANK) if refresh_context or not WORLD_BANK.is_file() else pd.read_csv(WORLD_BANK,encoding='utf-8-sig') # 下载或复用国家年份背景数据
events=attach_country_context(events,context,max_lag_years=3) # 附加事件当年或最近三年内的历史国家背景指标
ctif=extract_ctif_fire_service_capacity(CTIF_SOURCE,CTIF_CAPACITY) if CTIF_SOURCE.is_file() else pd.read_csv(CTIF_CAPACITY,encoding='utf-8-sig') # 从报告表1.13提取或复用国家消防服务能力背景
events=attach_fire_service_capacity(events,ctif) # 按ISO3为事件附加CTIF静态消防服务能力背景
registry=write_data_registry(DATA_REGISTRY) # 输出可纳入补充材料的数据源登记表
events.to_csv(FINAL,index=False,encoding='utf-8-sig') # 保存最终增强事件表
calendar=pd.read_csv(CALENDAR,encoding='utf-8-sig',parse_dates=['date']) # 读取病例交叉日期表
case_control=run_case_control_weather(calendar,CASE_CONTROL) if RUN_GEE else pd.DataFrame() # 批量补充GEE病例与对照日气象暴露
RUN_POWER=os.getenv('RUN_POWER','0')=='1' # 仅在显式开关后调用无需密钥但耗时的NASA POWER接口
power_case_control=download_nasa_power_case_control(calendar,POWER_CASE_CONTROL,POWER_CACHE) if RUN_POWER else pd.DataFrame() # 批量生成独立气象产品病例对照表
print('RUN_GEE=',RUN_GEE,'RUN_POWER=',RUN_POWER,'rows=',len(events),'context=',int(events['context_year'].notna().sum()),'ctif_capacity=',int(events['ctif_total_firefighters_per_100k'].notna().sum()),'sources=',len(registry)) # 显示两类气象提取状态与外部背景覆盖

GEE 1/2 HRF-0180 completed
GEE 2/2 HRF-0230 completed
病例交叉气象 1/5 HRF-0180
病例交叉气象 2/5 HRF-0129
病例交叉气象 3/5 HRF-0201
病例交叉气象 4/5 HRF-0230
病例交叉气象 5/5 HRF-0204
RUN_GEE= True RUN_POWER= False rows= 239 context= 237 ctif_capacity= 150 sources= 7
